# Exploring Chunking Strategies and Vector Databases

In this notebook, you will explore and compare chunking strategies for Retrieval-Augmented Generation (RAG), then evaluate how those choices influence retrieval outcomes.

Chunking is the process of splitting a document into smaller pieces (chunks). Each chunk can then be embedded, indexed, and retrieved. Strong chunking choices often improve retrieval relevance and answer quality in RAG systems.

This notebook combines step-by-step experiments with interactive dashboards so you can inspect chunks and retrieval results more efficiently.

## Recommended Hardware

This notebook can run on the following hardware or remote resources

✅ AMD Instinct™ Accelerators  
✅ AMD Radeon™ RX/PRO Graphics Cards  
✅ AMD EPYC™ Processors  
✅ AMD Ryzen™ (AI) Processors  

[![Open in AMD Developer Cloud](https://img.shields.io/badge/Open_in_AMD_Developer_Cloud-000000?logo=amd&logoSize=auto)](https://amd-ai-academy.com/github/AMDResearch/aup-ai-tutorials/blob/main/rag/01.rag-chunking.ipynb)  

## Software Environment

Install ROCm on your system

| Linux | Windows |
|-------|---------|
| [Install PyTorch](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/install/quick-start.html) | [PyTorch on Windows](https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/windows/install-pytorch.html)|
| [Install Docker container](https://amdresearch.github.io/aup-ai-tutorials//env/env-gpu.html) | |

## Goals

- Understand why chunking strategy matters in RAG.
- Compare simple, recursive, semantic, and LLM-based chunking methods on the same source text.
- Build vector databases from each chunking output and inspect retrieval behavior.
- Use interactive dashboards to compare chunk content side-by-side and explore similarity search results by strategy.

### Install Dependencies

Install the package dependencies needed for this notebook or series of notebooks.

First, get the `aup_config.py` script locally if needed. Then install the dependencies (`aup_setup()`). This step may take a few minutes and only needs to be done once.

In [ ]:
![ -f aup_config.py ] || wget https://raw.githubusercontent.com/AMDResearch/aup-ai-tutorials/refs/heads/main/rag/aup_config.py

In [ ]:
from aup_config import aup_setup
aup_setup()

## Let's Chunk

### Import Libraries

In [ ]:
import os
import requests
import re
import ipywidgets as widgets
from IPython.display import display

import langchain_core
from langchain_text_splitters import TokenTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS

### Download Document

We are going to download a copy of the [Vitis HLS User Guide](https://docs.amd.com/r/en-US/ug1399-vitis-hls) as content for the chunking experiments.

In [ ]:
base_url = 'https://docs.amd.com/api/khub/maps/JQtJoZLV908LbR5xokDqLw/attachments/9sLLuMlUume6oVQ1~HLSdg-JQtJoZLV908LbR5xokDqLw/content?download=true&Ft-Calling-App=ft%2Fturnkey-portal&Ft-Calling-App-Version=5.2.44'
download_dir = 'data_hls'
pdf_filename = 'vitis_hls_ug.pdf'

os.makedirs(download_dir, exist_ok=True)
if not os.path.isfile(os.path.join(download_dir, pdf_filename)):
    response = requests.get(base_url, stream=True)
    if response.status_code == 200:
        pdf_path = os.path.join(download_dir, pdf_filename)
        with open(pdf_path, 'wb') as file:
            file.write(response.content)

You will see that we get one document per PDF page because `mode` is set to `page` (default). If `mode` is set to `single`, we get one document containing all PDF content

In [ ]:
loader = PyPDFLoader(os.path.join(download_dir, pdf_filename), mode="page")
pdf_doc = loader.load()

Grab a content-dense section of the document

In [ ]:
docs = pdf_doc[23:38].copy()
len(docs)

Clean up headers and footers so we only process relevant content

In [ ]:
for doc in docs:
    doc.page_content = doc.page_content.replace('\nSend Feedback', '')
    doc.page_content = re.sub(r'^Vitis HLS User Guide\s*.*$', '',doc.page_content, flags=re.MULTILINE)
    doc.page_content = re.sub(r'^UG1399\s*.*$', '',doc.page_content, flags=re.MULTILINE)

Combine all page content into a single long string

In [ ]:
text = ""
for d in docs:
    text += d.page_content

print(f'{len(text)} character in the text')
docs_merged = [langchain_core.documents.base.Document(text)]

## Chunking Strategies

In this section, we compare four chunking strategies on the same text. We keep `chunk_size` and `chunk_overlap` fixed so the comparison is fair

For each strategy, the code follows the same flow: create a splitter, split `docs_merged`, then print basic statistics (number of chunks, total characters, and average chunk length)

In [ ]:
chunk_size = 400
chunk_overlap = 30

First, we define a helper function that summarizes chunk outputs

`chunking_stats` takes a list of chunks and returns: total chunks, total characters, and average chunk length. This makes it easy to compare splitters using the same metrics

In [ ]:
def chunking_stats(chunks: list[langchain_core.documents.base.Document]):
    total_documents = len(chunks)
    if total_documents < 1:
        return 0, 0, 0
    total_length = sum(len(chunk.page_content) for chunk in chunks)
    avg_length = total_length / total_documents
    return total_documents, total_length, avg_length

### Simple Chunking

Simple chunking splits text into fixed-size token windows with overlap. It is fast and predictable, but it may cut ideas in the middle.

In the next code cell, we create a `TokenTextSplitter`, split `docs_merged`, and print chunk statistics for this baseline approach.

In [ ]:
text_splitter_fixed = TokenTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
documents_fixed_split = text_splitter_fixed.split_documents(docs_merged)
res = chunking_stats(documents_fixed_split)
print(f"Number of chunks {res[0]}, total characters {res[1]:,}, average characters {res[2]:.2f}")

### Recursive Chunking

Recursive chunking tries larger boundaries first (for example paragraphs), then falls back to smaller boundaries only when needed. This usually keeps text units more natural

In the next code cell, we build a `RecursiveCharacterTextSplitter`, split the same input text, and print the same statistics for side-by-side comparison.

In [ ]:
text_splitter_recursive = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
documents_recursive_split = text_splitter_recursive.split_documents(docs_merged)
res = chunking_stats(documents_recursive_split)
print(f"Number of chunks {res[0]}, total characters {res[1]:,}, average characters {res[2]:.2f}")

### Semantic Chunking

Semantic chunking uses embeddings to split where meaning changes, instead of splitting only by size. This can improve retrieval quality, but chunk sizes are less uniform.

`breakpoint_threshold_type` controls how split points are detected from semantic distance values between neighboring text segments.
With `percentile`, the splitter cuts where distances are above a chosen percentile. Lower thresholds usually create more, smaller chunks; higher thresholds create fewer, larger chunks.
You can tune this behavior with `breakpoint_threshold_amount` to match your document style and retrieval needs.

In the next cells, we configure an embedding model, build a `SemanticChunker`, split the document, and print chunk statistics.

In [ ]:
base_url = "http://localhost:11434/v1/"
api_key = "abc-123"
embed_model = OpenAIEmbeddings(model="nomic-embed-text:v1.5", base_url=base_url, api_key=api_key, check_embedding_ctx_length=False)
semantic_splitter = SemanticChunker(embed_model, breakpoint_threshold_type="percentile")

In [ ]:
document_semantic_split = semantic_splitter.split_documents(docs_merged)
res = chunking_stats(document_semantic_split)
print(f"Number of chunks {res[0]}, total characters {res[1]:,}, average characters {res[2]:.2f}")

### LLM chunking

LLM chunking asks a language model to produce self-contained chunks based on meaning and structure. It can create highly coherent chunks, but it is slower and more expensive at indexing time.

In the next cells, we configure a chat model, send a chunking prompt, parse the model response into a list of chunks, convert each chunk into a `Document`, and print summary statistics

In [ ]:
llm = ChatOpenAI(model="qwen3.5:9b", base_url=base_url, api_key=api_key, temperature=0)

In [ ]:
chunking_prompt = ChatPromptTemplate.from_template("""
    You are an expert processing technical documents. Your task is to split the following document into
    meaningful chunks. Follow these guidelines:

    1. Chunks should contain complete ideas or concepts, each chunk should be understandable on its own
    2. More complex sections should be in smaller chunks
    3. Chunks should not be smaller than 15 words
    4. Remove headers that bring no value to the understanding of the content, such as "Chapter 1", "Section 2.3", etc
    5. Keep related information together
    6. Do not split code snippets or tables if possible
    7. Remove all `\n`

    DOCUMENT:
    {document}

    Return ONLY a valid list of strings, where each string is a chunk.
    Format your response as:
    [
      "chunk1 text",
      "chunk2 text",
      ...
    ]
    Do not forget to open and close the list with square brackets, and to put each chunk between double quotes.
    Do not include any explanations or additional text outside the list.
    """
)

In [ ]:
chunking_chain = chunking_prompt | llm
llm_response = chunking_chain.invoke({"document": docs_merged[0].page_content})

In [ ]:
llm_split = eval(llm_response.content)

In [ ]:
documents_llm_split = [langchain_core.documents.base.Document(chunk) for chunk in llm_split]
res = chunking_stats(documents_llm_split)
print(f"Number of chunks {res[0]}, total characters {res[1]:,}, average characters {res[2]:.2f}")

## Comparing Strategies

Use this section to quickly compare how two chunking strategies split the same source content.

The dashboard is intended to help you:
- see side-by-side chunk text for two different strategies,
- inspect a specific chunk index in each strategy,
- build intuition about how chunking choices may impact downstream retrieval quality.

This is a qualitative exploration tool, so focus on readability and structure differences between the two outputs.

In [ ]:
chunked_documents = {
    "simple": documents_fixed_split,
    "recursive": documents_recursive_split,
    "semantic": document_semantic_split,
    "LLM": documents_llm_split,
}

strategies = list(chunked_documents.keys())

left_strategy = widgets.Dropdown(options=strategies, value=strategies[0], description="Strategy:")
right_strategy = widgets.Dropdown(options=[], description="Strategy:")

left_index = widgets.IntSlider(value=0, min=0, max=0, step=1, description="Index:", continuous_update=False)
right_index = widgets.IntSlider(value=0, min=0, max=0, step=1, description="Index:", continuous_update=False)

left_info = widgets.HTML()
right_info = widgets.HTML()
left_output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="6px", width="100%"))
right_output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="6px", width="100%"))


def render_panel(strategy_dd, index_slider, info_html, output_widget):
    if strategy_dd.value is None:
        info_html.value = "<b>Strategy:</b> none"
        with output_widget:
            output_widget.clear_output()
            print("No strategy available.")
        return

    docs = chunked_documents[strategy_dd.value]
    index_slider.max = max(len(docs) - 1, 0)
    index_slider.value = min(index_slider.value, index_slider.max)

    with output_widget:
        output_widget.clear_output()
        if not docs:
            info_html.value = f"<b>Strategy:</b> {strategy_dd.value} | <b>Chunks:</b> 0"
            print("No chunks available.")
            return

        doc = docs[index_slider.value]
        info_html.value = (
            f"<b>Strategy:</b> {strategy_dd.value} | "
            f"<b>Chunks:</b> {len(docs)} | "
            f"<b>Index:</b> {index_slider.value} | "
            f"<b>Characters:</b> {len(doc.page_content):,}"
        )
        print(doc.page_content)


def sync_right_options():
    options = [strategy for strategy in strategies if strategy != left_strategy.value]
    current = right_strategy.value
    right_strategy.options = options
    right_strategy.value = current if current in options else (options[0] if options else None)


def refresh(_=None):
    render_panel(left_strategy, left_index, left_info, left_output)
    render_panel(right_strategy, right_index, right_info, right_output)


def on_left_change(_=None):
    sync_right_options()
    refresh()


left_strategy.observe(on_left_change, names="value")
right_strategy.observe(refresh, names="value")
left_index.observe(refresh, names="value")
right_index.observe(refresh, names="value")

on_left_change()

left_panel = widgets.VBox(
    [left_strategy, left_index, left_info, left_output],
    layout=widgets.Layout(width="50%"),
)
right_panel = widgets.VBox(
    [right_strategy, right_index, right_info, right_output],
    layout=widgets.Layout(width="50%"),
)

display(widgets.VBox([widgets.HTML("<h3>Chunking Strategy Comparison</h3>"), widgets.HBox([left_panel, right_panel], layout=widgets.Layout(width="100%"))]))

## Building a Vector Database

Now we create one vector database per chunking strategy. Then we run the same queries against each database to see how chunking choices affect retrieved results

In [ ]:
vectorstoredb_fixed = FAISS.from_documents(documents_fixed_split, embed_model)
vectorstoredb_recursive = FAISS.from_documents(documents_recursive_split, embed_model)
vectorstoredb_semantic = FAISS.from_documents(document_semantic_split, embed_model)
vectorstoredb_llm = FAISS.from_documents(documents_llm_split, embed_model)

Now let's query the four vector databases. Each one uses content split with a different chunking technique.

In [ ]:
query="What is one of the most important constructs in your program?"

In [ ]:
similarity_result_fixed = vectorstoredb_fixed.similarity_search(query)
similarity_result_fixed[0].page_content

In [ ]:
similarity_result_recursive = vectorstoredb_recursive.similarity_search(query)
similarity_result_recursive[0].page_content

In [ ]:
similarity_result_semantic = vectorstoredb_semantic.similarity_search(query)
similarity_result_semantic[0].page_content

In [ ]:
similarity_result_llm = vectorstoredb_llm.similarity_search(query)
similarity_result_llm[0].page_content

The number of similar results returned by the vector database is controlled by `k` in `similarity_search(query)`. The default is 4. Let's change it and compare the outputs.

In [ ]:
similarity_result_llm = vectorstoredb_llm.similarity_search(query, k=10)
for idx, sim in enumerate(similarity_result_llm):
    print(f'{idx=}: {sim.page_content}\n')

## Compare Similarity Search for Chunking Strategies

Use this dashboard to test one query and compare retrieval behavior across chunking strategies.

What you can do here:
- Enter a question in natural language and run retrieval
- Choose `k` to control how many similar chunks are retrieved
- Use the `Vector DB` selector to inspect results for a specific strategy
- Quickly switch strategies to compare the type and quality of returned context for the same query

This section is meant for practical comparison, helping you understand which chunking approach gives the most useful context for your use case.

In [ ]:
vectorstores = {
    "simple": vectorstoredb_fixed,
    "recursive": vectorstoredb_recursive,
    "semantic": vectorstoredb_semantic,
    "LLM": vectorstoredb_llm,
}

control_style = {"description_width": "initial"}

query_input = widgets.Text(value="What is a stream", description="Type your Query:", layout=widgets.Layout(width="70%"), style=control_style)
k_slider = widgets.IntSlider(value=1, min=1, max=10, step=1, description="k (number of similar documents):", continuous_update=False, style=control_style, layout=widgets.Layout(width="400px"))
vectorstore_dropdown = widgets.Dropdown(options=list(vectorstores.keys()), value="simple", description="Vector DB:", layout=widgets.Layout(width="320px"), style=control_style)
search_button = widgets.Button(description="Search", button_style="primary")
status = widgets.HTML()
result_output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="6px", width="70%"))

search_results = {name: [] for name in vectorstores}


def show_selected_results(_=None):
    selected_store = vectorstore_dropdown.value
    docs = search_results.get(selected_store, [])

    with result_output:
        result_output.clear_output()
        if not docs:
            print("No results for the selected vector database.")
            return

        for idx, doc in enumerate(docs, start=1):
            print(f"Result {idx}\n")
            print(doc.page_content)
            print("\n" + "-" * 80 + "\n")


def run_search(_=None):
    query = query_input.value.strip()
    if not query:
        status.value = "<b>Please enter a query.</b>"
        return

    for name, store in vectorstores.items():
        search_results[name] = store.similarity_search(query, k=k_slider.value)

    status.value = f"<b>Query:</b> {query} | <b>k:</b> {k_slider.value}"
    show_selected_results()

vectorstore_dropdown.observe(show_selected_results, names="value")
search_button.on_click(run_search)

display(widgets.VBox([query_input, widgets.HBox([k_slider, search_button]), vectorstore_dropdown, status, result_output]))

run_search()

## Exercises for the Reader

- Change `chunk_size` and `chunk_overlap`
- Explore other `breakpoint_threshold_type` in the `SemanticChunker`, e.g., `standard_deviation` or `interquartile`
- Try a different VectorDB such as [Chroma](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma) by replacing `FAISS` with `Chroma`
  - Do you note any difference in the quality of results?


## Conclusions

This notebook showed how chunking strategy directly affects what gets indexed and what gets retrieved in a RAG pipeline.

By comparing simple, recursive, semantic, and LLM-based chunking, you can see trade-offs between chunk structure, coherence, and retrieval behavior.

The interactive dashboards make this comparison practical: one helps inspect chunk content across strategies, and the other helps test a query and review retrieval results by vector database.

In practice, there is no universal best strategy—effective chunking depends on your document style, query patterns, and quality targets.

---

[AMD University Program](https://www.amd.com/aup)

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.

SPDX-License-Identifier: MIT